In [ ]:
# Map to 3D coordinates
# This script maps 2D coordinates to 3D coordinates based on a predefined mapping.

import pyvista as pv
import numpy as np
import xml.etree.ElementTree as ET
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.interpolate import griddata
import os
from scipy.spatial import Delaunay
import csv


def read_xml_point_file(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    # Find all <point> entries under the first <time_series>
    points = []
    for point in root.findall('.//point_set/time_series/point'):
        x = float(point.find('x').text)
        y = float(point.find('y').text)
        z = float(point.find('z').text)
        points.append([x, y, z])

    return np.array(points)
def split_mesh_by_elemTag(mesh, tag_name='elemTag', onpts=True, tag_values=[0,3]):
    """
    Splits a PyVista mesh into RV free wall and septum based on an element tag.

    Parameters:
    - mesh: pyvista.UnstructuredGrid or PolyData
    - tag_name: name of the cell array to use for splitting (default 'elemTag')

    Returns:
    - rv_freewall_mesh: mesh with elemTag == 0
    - septum_mesh: mesh with elemTag == 3
    """    
    if tag_name not in mesh.cell_data:
        print(f"Tag '{tag_name}' not found in mesh cell data. Returning all indices")
        rv_freewall_indices = np.arange(mesh.n_points) 
        septum_indices = np.arange(mesh.n_points) 
        return rv_freewall_indices, septum_indices
     
    if onpts: # If onpts is True, we will average the elemTag across points
        elem_tags = mesh.cell_data["elemTag"]
        point_tags = [[] for _ in range(mesh.n_points)]

        for cell_id in range(mesh.n_cells):
            for pt_id in mesh.get_cell(cell_id).point_ids:
                point_tags[pt_id].append(elem_tags[cell_id])

        # Average or first
        avg_point_tags = np.array([np.mean(t) if t else -1 for t in point_tags], dtype=int)
        #print(f'Average point tags number of points: {avg_point_tags.shape[0]}')
        mesh.point_data["elemTag"] = avg_point_tags
        
        rv_freewall_indices = np.where(avg_point_tags == tag_values[0])[0] # elemTag = 0
        septum_indices = np.where(avg_point_tags == tag_values[1])[0] # elemTag = 3
        
        #print(f'RV Free Wall number of points: {rv_freewall_indices.shape[0]}')
        #print(f'Septum number of points: {septum_indices.shape[0]}')

        # Remove overlapping indices from septum_indices
        septum_indices = np.setdiff1d(septum_indices, rv_freewall_indices, assume_unique=False)
        #print(f'Septum number of points after removing RV Free Wall: {septum_indices.shape[0]}')

    else: 
        #rv_freewall_mesh = mesh.threshold(value=0.1, scalars=tag_name, invert=True)  # elemTag == 0
        #septum_mesh = mesh.threshold(value=(2.9, 3.1), scalars=tag_name)            # elemTag == 3
        
        rv_freewall_indices = np.where(mesh.cell_data[tag_name] == tag_values[0])[0] # elemTag == 0
        septum_indices = np.where(mesh.cell_data[tag_name] == tag_values[1])[0]   # elemTag == 3
        # rv_freewall_mesh = mesh.extract_cells(rv_freewall_indices)
        # septum_mesh = mesh.extract_cells(septum_indices)

    if septum_indices.size == 0:
        print(f"Error: No points found with elemTag = {tag_values}")
        rv_freewall_indices = np.arange(mesh.n_points) 
        septum_indices = np.arange(mesh.n_points) 
        return rv_freewall_indices, septum_indices
     

    return rv_freewall_indices, septum_indices

def normalize(v):
    return v / np.linalg.norm(v)

def scale_angles(rv0_theta, rv1_theta, theta_rel):
    """
    Map angles from full 0-360 range with specific reference points:
    - rv0_theta maps to 210°
    - rv1_theta maps to 330° 
    - Remaining angles are compressed into the other parts of the circle
    
    Parameters:
    rv0_theta: First reference angle (maps to 210°)
    rv1_theta: Second reference angle (maps to 330°)
    theta_rel: Input angle(s) to be mapped (0-360°)
    
    Returns:
    theta_final: Mapped angle(s) in the range [0, 360)
    """
    
    # Calculate the angular span between reference points
    delta_theta = (rv1_theta - rv0_theta) % 360
    
    # Handle theta_rel as array or scalar
    theta_rel = np.atleast_1d(theta_rel)
    theta_final = np.zeros_like(theta_rel, dtype=float)
    
    # Target span: rv0→210°, rv1→330° gives us 120° for the reference range
    target_span = 120  # from 210° to 330°
    
    # Remaining angular space to compress: 360° - 120° = 240°
    remaining_space = 240
    remaining_input_space = 360 - delta_theta
    
    for i, theta in enumerate(theta_rel):
        # Normalize theta relative to rv0
        theta_norm = (theta - rv0_theta) % 360
        
        if theta_norm <= delta_theta:
            # Angle is within the reference range [rv0, rv1]
            # Map linearly to [210°, 330°]
            progress = theta_norm / delta_theta if delta_theta > 0 else 0
            theta_final[i] = 210 + progress * target_span
        else:
            # Angle is outside the reference range
            # Compress into the remaining space
            remaining_theta = theta_norm - delta_theta
            compression_ratio = remaining_space / remaining_input_space if remaining_input_space > 0 else 1
            compressed_angle = remaining_theta * compression_ratio
            
            # Map to the space from 330° to 210° (wrapping around)
            theta_final[i] = (330 + compressed_angle) % 360
    
    # Return scalar if input was scalar
    if len(theta_final) == 1:
        return theta_final[0]
    return theta_final

def convert_to_polar_2d(mesh_points, pts, scale_z=True, verbose=False, RV_sept=True):
    apex = np.array(pts[0])
    base = np.array(pts[1])
    rv0 = np.array(pts[2])
    rv1 = np.array(pts[3])

    z_axis = normalize(base - apex)
    relative_points = mesh_points - apex
    base_dist = np.linalg.norm(base - apex)

    z_components = np.dot(relative_points, z_axis)[:, np.newaxis] * z_axis
    radial_vectors = relative_points - z_components

    z_coords = np.dot(relative_points, z_axis)
    apex_z_location= np.dot(apex-apex, z_axis)
    base_z_location = np.dot(base-apex, z_axis)                         
                             
    if verbose: 
        print('Z coordinates range:', np.min(z_coords), np.max(z_coords))
        print('Apex z location:', apex_z_location, 'Base z location:', base_z_location)
    if scale_z:
        # Normalize z_coords to [0, 1]
        z_coords = (z_coords - np.min(z_coords)) / (np.max(z_coords) - np.min(z_coords))
    else: 
        # scale to apex base distance
        z_coords = z_coords / base_dist

    ref_vector = normalize(rv0 - apex - np.dot((rv0 - apex), z_axis) * z_axis)
    y_ref = normalize(np.cross(z_axis, ref_vector))
    x_ref = ref_vector

    x_comp = np.dot(radial_vectors, x_ref)
    y_comp = np.dot(radial_vectors, y_ref)
    theta = np.degrees(np.arctan2(y_comp, x_comp)) % 360
    if verbose:
        print('Theta range:', np.min(theta), np.max(theta))
    # Get rv0 and rv1 current angles
    rv0_vec = rv0 - apex - np.dot((rv0 - apex), z_axis) * z_axis
    rv1_vec = rv1 - apex - np.dot((rv1 - apex), z_axis) * z_axis

    rv0_x = np.dot(rv0_vec, x_ref)
    rv0_y = np.dot(rv0_vec, y_ref)
    rv1_x = np.dot(rv1_vec, x_ref)
    rv1_y = np.dot(rv1_vec, y_ref)

    rv0_theta = np.degrees(np.arctan2(rv0_y, rv0_x)) % 360
    rv1_theta = np.degrees(np.arctan2(rv1_y, rv1_x)) % 360

    # Compute angular separation
    delta_theta = (rv1_theta - rv0_theta) % 360
    scale_factor = 120 / delta_theta  # from angle 210-330 scale to be 120 degrees

    # Remap full circle
    theta_rel = (theta - rv0_theta) % 360
    # If theta_rel is >300, then subtract 360
    theta_rel[theta_rel> 360] -= 360

    # If for the RV septal face 
    if RV_sept: 
        theta_scaled = (theta_rel * scale_factor) % 360
        theta_final = (theta_scaled + 210) % 360 # rotate to start at 210 
    else: # if for the full LV mapping 
    #     if theta_rel< delta_theta: 
    #         theta_scaled = (theta_rel * scale_factor) % 360
    #     else    
    #         scale_factor2 = 240/(360-delta_theta)
    #         theta_scaled = (theta_rel * scale_factor2)

    #     theta_final=theta_scaled + 120
    #     theta_final[theta_final> 360] -= 360

        theta_final=scale_angles(rv0_theta, rv1_theta, theta_rel)
        
    if verbose:
        print('rv0 theta:', rv0_theta, 'rv1 theta:', rv1_theta)
        print('Min theta:', np.min(theta_rel), 'Max theta:', np.max(theta_rel))
        print(f"rv0 θ = {rv0_theta:.2f}° → 210°, rv1 θ = {rv1_theta:.2f}° → 330°")
        print(f"Angular separation = {delta_theta:.2f}°, scale = {scale_factor:.4f}")
        print(f"Theta range: {np.min(theta_final):.2f}° to {np.max(theta_final):.2f}°")

    polar_coords = np.vstack((theta_final, z_coords)).T
    return polar_coords
def create_segment_mask(angle0, angle1, radius0, radius1, theta_grid, radius_grid):
    """
    Create a boolean mask for a segment defined by angular and radial bounds
    
    Parameters:
    angle0, angle1: Angular bounds in degrees
    radius0, radius1: Radial bounds (inner, outer)
    theta_grid, radius_grid: Coordinate grids (in degrees and normalized radius)
    
    Returns:
    mask: Boolean array where True indicates points within the segment
    """
    
    # Handle wraparound case (e.g., 330-360 and 0-30 for anterior segments)
    if angle0 > angle1:  # This shouldn't happen with our corrected data structure
        angle_mask = (theta_grid >= angle0) | (theta_grid <= angle1)
    else:
        angle_mask = (theta_grid >= angle0) & (theta_grid <= angle1)
    
    # Radial mask
    radius_mask = (radius_grid >= radius0) & (radius_grid <= radius1)
    
    # Combine masks
    mask = angle_mask & radius_mask
    
    return mask


def map_AHA_regions(csv_file, septum_mesh, field_str=None): 
    """
    Map AHA (American Heart Association) 17-segment model values to mesh points.
    
    Parameters:
    csv_file: Path to CSV file with region values, or None for default mapping
    septum_mesh: Mesh object with point_data containing 'z_coords' and 'theta'
    field_str: String identifier for the field being mapped
    
    Returns:
    septum_mesh: Updated mesh with new field containing mapped values
    """
    
    # Radial boundaries (normalized: 0 = center, 1 = edge)
    apical_radius = 0.33   # Inner third (apex)
    mid_radius = 0.67      # Middle third  
    basal_radius = 1.0     # Outer third (base)
    
    # AHA 17-segment model definition
    # Note: Segments 1, 7, and 13 cross the 0°/360° boundary
    segment_labels = {
        # Basal segments (1-6)
        1: [(0, 30, mid_radius, basal_radius), (330, 360, mid_radius, basal_radius)],    # Basal anterior
        2: [(270, 330, mid_radius, basal_radius)],   # Basal anteroseptal  
        3: [(210, 270, mid_radius, basal_radius)],   # Basal inferoseptal
        4: [(150, 210, mid_radius, basal_radius)],   # Basal inferior
        5: [(90, 150, mid_radius, basal_radius)],    # Basal inferolateral
        6: [(30, 90, mid_radius, basal_radius)],     # Basal anterolateral
        
        # Mid-cavity segments (7-12)
        7: [(0, 30, apical_radius, mid_radius), (330, 360, apical_radius, mid_radius)],  # Mid anterior
        8: [(270, 330, apical_radius, mid_radius)],   # Mid anteroseptal
        9: [(210, 270, apical_radius, mid_radius)],   # Mid inferoseptal  
        10: [(150, 210, apical_radius, mid_radius)],  # Mid inferior
        11: [(90, 150, apical_radius, mid_radius)],   # Mid inferolateral
        12: [(30, 90, apical_radius, mid_radius)],    # Mid anterolateral
        
        # Apical segments (13-16)
        13: [(315, 360, 0, apical_radius), (0, 45, 0, apical_radius)],  # Apical anterior
        14: [(225, 315, 0, apical_radius)],   # Apical septal
        15: [(135, 225, 0, apical_radius)],   # Apical inferior  
        16: [(45, 135, 0, apical_radius)],    # Apical lateral
    }

    def create_segment_mask(angle0, angle1, radius0, radius1, theta, z_coords):
        """Create boolean mask for points within specified angular and radial bounds"""
        
        # Handle angle wrapping around 0°/360° boundary
        if angle0 <= angle1:
            angle_mask = (theta >= angle0) & (theta <= angle1)
        else:  # wraps around 0°/360°
            angle_mask = (theta >= angle0) | (theta <= angle1)
        
        # Create radial mask
        radius_mask = (z_coords >= radius0) & (z_coords <= radius1)
        
        # Combine masks
        return angle_mask & radius_mask

    # Initialize output array with NaN
    assigned_values = np.full(septum_mesh.n_points, np.nan)
    
    # Get mesh coordinates
    z_coords = septum_mesh.point_data['z_coords']
    theta = septum_mesh.point_data['theta']
    
    # Load data from CSV file or use defaults
    if csv_file is not None: 
        try:
            with open(csv_file, 'r') as f:
                reader = csv.reader(f)
                next(reader)  # Skip header
                region_stats = [row for row in reader]
                AHA_regions = [int(row[0]) for row in region_stats]
                values = [float(row[1]) for row in region_stats]  # mean values for each region
                field = f'mean_{field_str}'
                
        except FileNotFoundError: 
            print(f'File not found: {csv_file}')
            return septum_mesh
            
    else: 
        # Default mapping: assign region number as value
        field = 'AHA_mapping'
        AHA_regions = list(range(1, 17))  # [1, 2, 3, ..., 16]
        values = list(range(1, 17))       # [1, 2, 3, ..., 16]

    # Process each AHA region
    for i, region in enumerate(AHA_regions):
        if region not in segment_labels:
            print(f'Warning: Region {region} not found in segment definitions')
            continue
            
        segment_ranges = segment_labels[region]
        mean_value = values[i]
        
        # Process each angular range for this segment (handles wraparound cases)
        for angle0, angle1, radius0, radius1 in segment_ranges:
            region_mask = create_segment_mask(angle0, angle1, radius0, radius1, theta, z_coords)
            
            # Assign value to all points in this region
            if np.any(region_mask):
                assigned_values[region_mask] = mean_value

    # Add the new field to mesh
    septum_mesh.point_data[field] = assigned_values
    
    return septum_mesh
def AHAmap_3D(mainpath, xml_file, refmesh, output_file,flip_flag=True, verbose=False): 

    try:
        points_array = read_xml_point_file(xml_file)
        if verbose: 
            print(f'Points array: {points_array}')
        if flip_flag: 
        # Flip 
            points_array[:,0]=-points_array[:,0]
            points_array[:,1]=-points_array[:,1]

        point_cloud = pv.PolyData(points_array)
        point_cloud.save(f'{mainpath}/1_seg/ap_mv_2rv_Siemens.vtk')
    except FileNotFoundError:
        print(f"XML file not found for case {casenum:02d}. Skipping this case.")
        return    
    
    print('Splitting mesh into LV indices')
    _, septum_indices = split_mesh_by_elemTag(refmesh, tag_name='elemTag',onpts=True, tag_values=[0, 1])


    septum_mesh = refmesh.extract_points(septum_indices, adjacent_cells=False)
    if verbose: 
        print(refmesh)
        print(f'Septum indices: {septum_indices}')
        septum_meshfile=f'{mainpath}/test.vtk'
        print(f'Save septum mesh to {septum_meshfile}')
        septum_mesh.save(septum_meshfile)

    septum_points = convert_to_polar_2d(septum_mesh.points,  points_array,scale_z=False, verbose=True, RV_sept=False)
    if verbose: 
        print(f'converted septum points: {septum_points}')
    septum_mesh.point_data['theta'] = septum_points[:, 0]
    septum_mesh.point_data['z_coords'] = septum_points[:, 1]

    septum_mesh=map_AHA_regions(csv_file=None, septum_mesh=septum_mesh)

    
    print(f'Saving septum mesh for case {casenum:02d} to {output_file}')
    septum_mesh.save(output_file)
    return

if __name__ == "__main__":   
    for casenum in [1]: 
        if casenum in [3, 5, 12]: 
            continue
        mainpath=f'/data_fast/Work/MS_HF_cohort/data/{casenum:02d}'
        # LV mesh
        LV_refmesh=pv.read(f'{mainpath}/LV.vtk')
        LV_output_file= f'{mainpath}/1_seg/LV_AHA.vtk'
    
        # RV septum mesh 
        RV_refmesh=pv.read(f'{mainpath}/1_seg/RV_smooth_flip.vtk')
        RV_output_file= f'{mainpath}/1_seg/sept_AHA.vtk'
        
        

        xml_file = f'{mainpath}/1_seg/ap_mv_2rv_Siemens.mps'

        #AHAmap_3D(xml_file, LV_refmesh, LV_output_file, flip_flag=True)
        AHAmap_3D(mainpath, xml_file, RV_refmesh, RV_output_file, flip_flag=True, verbose=True)
        

Points array: [[ 73.4769  71.7324 360.213 ]
 [ 25.1018 135.253  377.43  ]
 [ 17.22   110.552  338.937 ]
 [ 44.8454  96.6719 408.836 ]]
Splitting mesh into LV indices
Error: No points found with elemTag = [0, 1]
PolyData (0x712a7ed8aa40)
  N Cells:    15385
  N Points:   7918
  N Strips:   0
  X Bounds:   -5.222e+01, 3.373e+00
  Y Bounds:   -1.166e+02, -6.719e+01
  Z Bounds:   3.335e+02, 4.166e+02
  N Arrays:   4
Septum indices: [   0    1    2 ... 7915 7916 7917]
Save septum mesh to /data_fast/Work/MS_HF_cohort/data/01/test.vtk
Z coordinates range: 11.431149310289046 79.2477571952144
Apex z location: 0.0 Base z location: 81.67889576487919
Theta range: 0.009569639710022153 359.9977277580032
rv0 theta: 1.0713887174338752e-15 rv1 theta: 137.4463063146196
Min theta: 0.009569639710021081 Max theta: 359.9977277580032
rv0 θ = 0.00° → 210°, rv1 θ = 137.45° → 330°
Angular separation = 137.45°, scale = 0.8731
Theta range: 200.54° to 327.34°
converted septum points: [[2.00658395e+02 3.04148197e-0

In [ ]:
def get_mesh_output(device, length, collision_field= 'Combined_Collision'): 
    all_interp_fields_RV_freewall = []
    all_interp_fields_RV_septum = []
    for casenum in range(1,14): #[1:13]:
        if casenum in [3, 5, 12]: # Skip cases with no data
            print(f'Skipping case {casenum:02d} due to no data.')
            continue
        mainpath=f'/data_fast/Work/MS_HF_cohort/data/{casenum:02d}'
        output_file= f'{mainpath}/1_seg/RV_septum_polar.vtk'
        
        if collision_field == 'Combined_Collision':
            images_outpath=f'/data_fast/Work/MS_HF_cohort/images/{device}/'
            add_legend=True
        else: 
            images_outpath=f'/data_fast/Work/MS_HF_cohort/images/{device}/{collision_field}/'
            add_legend=False
        
        xml_file = f'{mainpath}/1_seg/ap_mv_2rv_Siemens.mps'
        try:
            points_array = read_xml_point_file(xml_file)
            print(points_array)
        except FileNotFoundError:
            print(f"XML file not found for case {casenum:02d}. Skipping this case.")
            continue    
                
        # Read the septum mesh if it exists, otherwise create it
        if os.path.exists(output_file):
            septum_mesh = pv.read(output_file)
            print(f'Loading existing septum mesh for case {casenum:02d} from {output_file}')
            
        else: 
            init_mesh=f'{mainpath}/1_seg/RV_smooth.vtk'
            print(f'Processing case {casenum:02d} with mesh {init_mesh}')
            refmesh=pv.read(init_mesh)
            rv_freewall_indices, septum_indices = split_mesh_by_elemTag(refmesh, tag_name='elemTag',onpts=True)
            septum_mesh = refmesh.extract_points(septum_indices, adjacent_cells=False)

        # Process the septum mesh
        septum_points = convert_to_polar_2d(septum_mesh.points, points_array)
        
        septum_mesh.point_data['theta'] = septum_points[:, 0]
        septum_mesh.point_data['z_coords'] = septum_points[:, 1]

        csv_file=f'{images_outpath}/sum_plots/RV_septum_polar_sum_{device}_L{length}.csv'
        #print(f'Loading existing CSV file for case {casenum:02d} from {csv_file}')
        field_str= f'{device}{length}_{collision_field.replace('_Collision', '')}'
        septum_mesh=map_AHA_regions(csv_file, septum_mesh, field_str=field_str)

        print(f'Saving septum mesh for case {casenum:02d} to {mainpath}/1_seg/RV_septum_polar.vtk')
        septum_mesh.save(output_file)
        #return septum_mesh
        # add in the points from the xml file 
        #lm_points= convert_to_polar_2d(points_array, points_array)
        return 
def collmap_3D():
    # Map to AHA
    # AveirVR 38mm, AveirAR 32.2mm, Micra 25.9mm
    ########################################################
    for device in ['AveirVR', 'AveirAR', 'Micra']: 
        if device == 'AveirVR':
            init_len= 38
        elif device == 'AveirAR': 
            init_len= 32.2
        elif device == 'Micra': 
            init_len = 25.9
        else: 
            print(f'WARNING - DEVICE: {device} is not AveirVR, AveirAR or Micra')        


        for length in [init_len]: #, 15, 20, 25, 30, 35, 40, 45]:  #[22, 27, 32, 37, 42]: 
            for collision_field in ['RV_Collision', 'TV_Collision', 'PM_Collision', 'Combined_Collision']: #['Combined_Collision']: #
                get_mesh_output(device, length, collision_field=collision_field)
if __name__ == "__main__":
    # To map the averaged collision map value on the AHA mapped onto the 3D mesh
    collmap_3D()

SyntaxError: incomplete input (1813473880.py, line 75)

In [158]:
# Visualize the mesh
import numpy as np
import pyvista as pv    
import vtk
from vtkmodules.vtkRenderingAnnotation import vtkScalarBarActor
from vtkmodules.vtkCommonCore import vtkStringArray, vtkDoubleArray
import matplotlib as plt
def set_scalar_bar_mid_labels(plotter, n_labels=5):
    """
    Replace scalar bar tick labels with midpoints between bins.
    
    Parameters
    ----------
    plotter : pyvista.Plotter
        The plotter containing the scalar bar.
    n_labels : int
        Number of midpoint labels to show.
    """
    if not plotter.scalar_bars:
        raise RuntimeError("No scalar bar found in the plotter.")
    
    # Grab the first scalar bar actor
    scalar_bar: vtkScalarBarActor = list(plotter.scalar_bars.values())[0]
    
    # Get scalar range
    #vmin, vmax = scalar_bar.GetLookupTable().GetRange()
    #print(vmin, vmax)
    # Compute bin centers
    #edges = np.linspace(1, 16, n_labels + 1)
    #centers = 0.5 * (edges[:-1] + edges[1:])
    centers = np.arange(1.4, 17.4, 1) # arange goes to 1 below
    #print(centers)
    # Convert to vtk array
    ticks = vtkDoubleArray()
    for c in centers:
        ticks.InsertNextValue(float(c))
    #print(ticks)
    #print(centers)
    # Apply custom tick positions
    scalar_bar.SetCustomLabels(ticks)
    scalar_bar.SetUseCustomLabels(True)
    scalar_bar.SetNumberOfLabels(len(centers))
    scalar_bar.SetLabelFormat('%.0f')
    #scalar_bar.label_text_property.SetFontSize(14)
    #scalar_bar.title_text_property.SetFontSize(16)

def mycolormap(): 

    colors = [
        "#4da6ff",  # your blue
        "#66b38c",  # your teal
        "#ff9933",  # your orange
        "#ffd92f",  # yellow
        "#a6d854",  # lime green
        "#8da0cb",  # soft blue
        "#e78ac3",  # pink
        "#fc8d62",  # salmon
        "#b3b3b3",  # gray
        "#1b9e77",  # strong green
        "#d95f02",  # strong orange
        "#7570b3",  # purple
        "#e7298a",  # magenta
        "#66a61e",  # dark green
        "#e6ab02",  # mustard
        "#a6761d",  # brown
    ]

    
    cmap = ListedColormap(colors[:16], name="colorblind_rainbow")
    return cmap
def set_camera_position(plotter):
    # 
    # view = GetActiveView()
    # print("CameraPosition:", view.CameraPosition)
    # print("CameraFocalPoint:", view.CameraFocalPoint)
    # print("CameraViewUp:", view.CameraViewUp)
    # print("CameraParallelScale:", view.CameraParallelScale)

    plotter.camera_position = [
        (152.42287108591592, 123.59827412877627, 656.7684240299902), 
        (-29.384029209520445, -93.8310364745493, 379.2673664416711),
        (0.6384333843111966, -0.7506886833483885, 0.16990972452927794)
    ]
    plotter.enable_parallel_projection()
    plotter.camera.parallel_scale = 102.66216262784093
def plot_meshes(plotter, biv_masked_mesh,septum_mesh,scalars='mean_AveirVR38_RV',screenshot_path=None, cmap='PuBu', clim=[0, 100], n_colors=20,  nan_color=[0,0,0,0], scalar_bar_title='RV Septum Mean Values'): 
    
#cmap='coolwarm',
    #shifted_data=scalars+0.5
    plotter.add_mesh(septum_mesh, scalars=scalars, cmap=cmap, show_edges=False,                 
                     name=scalar_bar_title, clim=clim, n_colors=n_colors, 
                     show_scalar_bar=True, nan_color=nan_color, interpolate_before_map=False, opacity=1)

    set_scalar_bar_mid_labels(plotter, n_labels=n_colors)   
    plotter.add_mesh(biv_masked_mesh, color='lightgray', opacity=0.1, show_edges=False, name='BiV_Masked')
    set_camera_position(plotter)

    return plotter


def get_collision_map_screenshots():     

    for case in [1]: #,2,4,6,7,8,9,10,11,13]: #range(1,14): #[1:13]:
        mainpath=f'/data_fast/Work/MS_HF_cohort/data/{case:02d}'  
        # Read the septum mesh
        septum_mesh = pv.read(f'{mainpath}/1_seg/RV_septum_polar.vtk')
        # Read in BiV_masked mesh 
        biv_masked_mesh = pv.read(f'{mainpath}/1_seg/BiV_masked.vtk') 

        for device in ['Micra']: # 'AveirVR', 'AveirAR', 
            if device == 'AveirVR':
                init_len= 38
            elif device == 'AveirAR': 
                init_len= 32.2
            elif device == 'Micra': 
                init_len = 25.9
            else: 
                print(f'WARNING - DEVICE: {device} is not AveirVR, AveirAR or Micra')        

                #, RV_Collision, TV_Collision, 'PM_Collision', 'Combined_Collision'
            
            for length in [init_len]: #, 15, 20, 25, 30, 35, 40, 45]:  #[22, 27, 32, 37, 42]: 
                for collision_field in ['Combined_Collision']: #['Combined_Collision']: #
                    field_str= f'{device}{length}_{collision_field.replace("_Collision", "")}'
                    print(f'Plotting for device {device}, length {length}, collision field {collision_field}')
                    screenshot_path = f'/data_fast/Work/MS_HF_cohort/images/3D_output/case{case:02d}_RV_septum_polar_sum_{field_str}.png'
                    plot_meshes(biv_masked_mesh, septum_mesh, scalars=f'mean_{field_str}', screenshot_path=screenshot_path)

        #plot_meshes(biv_masked_mesh,septum_mesh,scalars='mean_AveirVR38_RV',screenshot_path=None)

def get_AHA_map_screenshots(mainpath, septum_file='1_seg/LV_AHA.vtk', scalars='AHA_mapping', screenshot_path=None, addpm=True, addtv=True, addrv=False, addcy=False, addlv=False, addCS=False):     
    
        # Read in BiV_masked mesh 
        biv_masked_mesh = pv.read(f'{mainpath}/2_motion/BiV_masked_flip.vtk') 
        if addtv: 
            tv_mesh=pv.read(f'{mainpath}/2_motion/TV_flip.vtk')
        if addpm: 
            pm_mesh=pv.read(f'{mainpath}/2_motion/PM_flip.vtk')
        if addrv: 
            rv_mesh=pv.read(f'{mainpath}/2_motion/RV_smooth_flip.vtk')
        if addcy:
            cy_mesh=pv.read(f'{mainpath}/2_motion/cylinder200.vtp')
        if addlv: 
            if septum_file is None: 
                biv_masked_mesh = pv.read(f'{mainpath}/2_motion/BiV_masked_flip.vtk') 
            else: 
                lv_mesh=pv.read(f'{mainpath}/1_seg/LV_AHA.vtk')
            
        if addCS: 
            CS_upper=pv.read(f'{mainpath}/1_seg/CS_upper.vtk')

        ####################################################    
        plotter = pv.Plotter()
        set_camera_position(plotter) 
        ### PLOT THE LV AHA mesh with the biv mesh 
        if addlv: 
            if septum_file is not None: 
                print(f'add the {mainpath}/1_seg/LV_AHA.vtk file First')

                plotter.add_mesh(biv_masked_mesh, color='lightgray', opacity=0.1, show_edges=False, name='BiV_Masked')
                lv_mesh[scalars]=lv_mesh[scalars]+0.4        
                plotter.add_mesh(lv_mesh, scalars=scalars, cmap='coolwarm', show_edges=False,                 
                    clim=[1,17], n_colors=16, show_scalar_bar=False, nan_color='lightgray', interpolate_before_map=False, opacity=0.3)
                septum_mesh = pv.read(f'{mainpath}/{septum_file}')         
                plotter.add_mesh(septum_mesh, scalars=scalars, cmap='coolwarm', show_edges=False,                 
                    clim=[1,17], n_colors=16, show_scalar_bar=False, nan_color=[0,0,0,0], interpolate_before_map=False, opacity=1)
                
            else: # Add the biv mesh when we don't have a septum_file 
                plotter.add_mesh(biv_masked_mesh, opacity=0.1, color="white")
        # Read the septum mesh
        else: 
            if septum_file is not None: 
                septum_mesh = pv.read(f'{mainpath}/{septum_file}')            
                septum_mesh[scalars]=septum_mesh[scalars]+0.4     
                plotter=plot_meshes(plotter, biv_masked_mesh, septum_mesh, scalars=scalars, screenshot_path=None, cmap='coolwarm', clim=[1, 17], n_colors=16, scalar_bar_title='AHA regions', nan_color=[0.8,0.8,0.8,0.1])
            
                
        if addpm: 
            plotter.add_mesh(pm_mesh, opacity=0.5, color="#ff9933")
        if addtv: 
            plotter.add_mesh(tv_mesh, opacity=0.5, color="#4da6ff") 
        if addrv: 
            plotter.add_mesh(rv_mesh, opacity=0.2, color="#66b38c")
        if addcy: 
            plotter.add_mesh(cy_mesh, opacity=0.8, color="white") 
        if addCS: 
            plotter.add_mesh(CS_upper,
					 scalars="elemTag",
					 show_edges=False,
					 cmap=plt.cm.get_cmap("coolwarm",8),
					 line_width=7,
					 style='wireframe',
					 render_lines_as_tubes=True)

        if screenshot_path is not None: 
            plotter.screenshot(screenshot_path)
            print(f'Screenshot saved to {screenshot_path}')
            plotter.close()  # Close without showing
        else: 
            # Onlt show the plot if no screenshot path is provided
            plotter.show()
if __name__ == "__main__":
    #get_collision_map_screenshots()
    # Figure 1A 

    #for case in [1]: #,2,4,6,7,8,9,10,11,13]: #range(1,14): #[1:13]:
    mainpath=f'/data_fast/Work/MS_HF_cohort/data/01'  
    #screenshot_path = f'/data_fast/Work/MS_HF_cohort/images/3D_output/case01_BiV_masked_LV_AHA.png'
    #screenshot_path=None
    #get_AHA_map_screenshots(mainpath, septum_file='1_seg/LV_AHA.vtk', scalars='AHA_mapping', screenshot_path = screenshot_path, addpm=True, addtv=True, addrv=False)

    #screenshot_path = f'/data_fast/Work/MS_HF_cohort/images/3D_output/case01_BiV_masked_RV_AHA.png'
    #get_AHA_map_screenshots(mainpath, septum_file='1_seg/sept_AHA.vtk', scalars='AHA_mapping', screenshot_path = screenshot_path, addpm=True, addtv=True, addrv=True, addcy=True, addlv=True, addCS=False)
    
    #screenshot_path = f'/data_fast/Work/MS_HF_cohort/images/3D_output/case01_BiV_all_structures.png'
    #get_AHA_map_screenshots(mainpath, septum_file=None, scalars='AHA_mapping', screenshot_path = screenshot_path, addpm=True, addtv=True, addrv=False, addcy=False, addlv=True, addCS=False)
    
    screenshot_path = f'/data_fast/Work/MS_HF_cohort/images/3D_output/case01_BiV_masked_RV_AHA.png'
    get_AHA_map_screenshots(mainpath, septum_file='1_seg/sept_AHA.vtk', scalars='AHA_mapping', screenshot_path = screenshot_path, addpm=True, addtv=True, addrv=True, addcy=True, addlv=True, addCS=False)
    


add the /data_fast/Work/MS_HF_cohort/data/01/1_seg/LV_AHA.vtk file First
Screenshot saved to /data_fast/Work/MS_HF_cohort/images/3D_output/case01_BiV_masked_RV_AHA.png


In [1]:
import pyvista as pv
import numpy as np


def extract_septum_from_rv_mesh(input_file, output_file, septum_tag=3):
    """
    Extract septum region from RV mesh based on elemTag value.
    Works whether elemTag is stored in point_data or cell_data.
    
    Parameters:
    input_file: Path to input VTK file (e.g., 'RV_smooth_flip.vtk')
    output_file: Path to output VTK file (e.g., 'RV_septum_polar.vtk')
    septum_tag: Value in elemTag field that identifies septum (default: 3)
    """
    
    # Load the mesh
    print(f"Loading mesh from: {input_file}")
    rv_mesh = pv.read(input_file)
    
    # Display mesh info
    print(f"Original mesh:")
    print(f"  Points: {rv_mesh.n_points}")
    print(f"  Cells: {rv_mesh.n_cells}")
    print(f"  Point data arrays: {list(rv_mesh.point_data.keys())}")
    print(f"  Cell data arrays:  {list(rv_mesh.cell_data.keys())}")
    
    # Check where elemTag exists
    elem_source = None
    if 'elemTag' in rv_mesh.point_data:
        elem_source = 'point'
        elem_tags = rv_mesh.point_data['elemTag']
    elif 'elemTag' in rv_mesh.cell_data:
        elem_source = 'cell'
        elem_tags = rv_mesh.cell_data['elemTag']
    else:
        print("Error: 'elemTag' field not found in point_data or cell_data")
        return None
    
    print(f"\nUsing elemTag from: {elem_source}_data")
    unique_tags = np.unique(elem_tags)
    print(f"Unique elemTag values: {unique_tags}")
    
    # Count elements by tag
    for tag in unique_tags:
        count = np.sum(elem_tags == tag)
        print(f"  Tag {tag}: {count} {'points' if elem_source=='point' else 'cells'}")
    
    # Extraction
    print("\nExtracting septum mesh...")
    if elem_source == 'point':
        # Threshold on point_data
        #septum_mesh = rv_mesh.threshold(septum_tag - 0.5, septum_tag + 0.5, scalars='elemTag')
        septum_mesh = rv_mesh.threshold(value=(septum_tag - 0.5, septum_tag + 0.5), scalars='elemTag')
    else:
        # Threshold on cell_data
        #septum_mesh = rv_mesh.threshold(septum_tag - 0.5, septum_tag + 0.5, scalars='elemTag', preference='cell')
         septum_mesh = rv_mesh.threshold(value=(septum_tag - 0.5, septum_tag + 0.5), scalars='elemTag', preference='cell')
    if septum_mesh.n_points == 0:
        print("Error: Extraction failed (no points).")
        return None
    
    # Verify
    print(f"\nFinal septum mesh:")
    print(f"  Points: {septum_mesh.n_points}")
    print(f"  Cells: {septum_mesh.n_cells}")
    if 'elemTag' in septum_mesh.point_data:
        print(f"  Extracted point elemTags: {np.unique(septum_mesh.point_data['elemTag'])}")
    if 'elemTag' in septum_mesh.cell_data:
        print(f"  Extracted cell elemTags: {np.unique(septum_mesh.cell_data['elemTag'])}")
    
    # Save
    print(f"\nSaving septum mesh to: {output_file}")

    septum_mesh = septum_mesh.extract_surface()  # Converts to PolyData
    septum_mesh.save(output_file, binary=False)

    print("✓ Septum mesh saved successfully!")
    
    return septum_mesh

# Main execution
if __name__ == "__main__":
    mainpath='/data_fast/Work/MS_HF_cohort/data/01/'
    input_file = f"{mainpath}/1_seg/RV_smooth_flip.vtk"
    output_file = f"{mainpath}/1_seg/RV_smooth_flip.vtk"
    
    # Extract the septum mesh
    septum_mesh = extract_septum_from_rv_mesh(input_file, output_file, septum_tag=3)
    
    if septum_mesh is not None:
        print("\n" + "="*50)
        print("SUCCESS! Septum mesh extracted and saved.")
        print("="*50)
        
        # Optional: Load both meshes for comparison
        rv_mesh = pv.read(input_file)
        
        # Uncomment the line below if you want to visualize
        # visualize_extraction(rv_mesh, septum_mesh)
    
    else:
        print("\n" + "="*50)
        print("FAILED! Could not extract septum mesh.")
        print("="*50)

# Quick one-liner version:
print("\nQUICK ONE-LINER VERSION:")
print("rv_mesh = pv.read('RV_smooth_flip.vtk')")
print("septum_mesh = rv_mesh.threshold(2.5, 3.5, scalars='elemTag')")
print("septum_mesh.save('RV_septum_polar.vtk')")

Loading mesh from: /data_fast/Work/MS_HF_cohort/data/01//1_seg/RV_smooth_flip.vtk
Original mesh:
  Points: 7918
  Cells: 15385
  Point data arrays: ['vtkOriginalPointIds']
  Cell data arrays:  ['elemTag', 'vtkOriginalCellIds']

Using elemTag from: cell_data
Unique elemTag values: [3]
  Tag 3: 15385 cells

Extracting septum mesh...

Final septum mesh:
  Points: 7918
  Cells: 15385
  Extracted cell elemTags: [3]

Saving septum mesh to: /data_fast/Work/MS_HF_cohort/data/01//1_seg/RV_smooth_flip.vtk
✓ Septum mesh saved successfully!

SUCCESS! Septum mesh extracted and saved.

QUICK ONE-LINER VERSION:
rv_mesh = pv.read('RV_smooth_flip.vtk')
septum_mesh = rv_mesh.threshold(2.5, 3.5, scalars='elemTag')
septum_mesh.save('RV_septum_polar.vtk')


In [ ]:
def visualise_fascicles(meshname,
						title,
						figname,
						camera_settings_file,
						fig_w=500,
						fig_h=500):
	print("Visualising fascicles...")
	if not os.path.exists(meshname+".pts") or not os.path.exists(meshname+".elem"):
		raise Exception("Cannot find mesh "+meshname+" in CARP format")
	
	if not os.path.exists(camera_settings_file):
		raise Exception("The camera file you provided does not exist")
	camera_settings = load_json(camera_settings_file)
	plotter = pv.Plotter(off_screen=True)
	plotter.background_color = 'white'
	plt_myo_pkj = carp_to_pyvista_myo_pkj(meshname)
	plt_fascicles = extract_tagged_cells(plt_myo_pkj, "tags", 150,157)
	plotter.add_mesh(plt_fascicles,
					 scalars="tags",
					 show_edges=False,
					 cmap=plt.cm.get_cmap("coolwarm",8),
					 line_width=7,
					 style='wireframe',
					 render_lines_as_tubes=True)
	if title is not None:
		plotter.add_title(title,
						  font_size=12,
						  font="arial",
						  color="black")
	plotter.camera.azimuth = camera_settings["azimuth"]
	plotter.camera.elevation = camera_settings["elevation"]
	plotter.camera.roll = camera_settings["roll"]	
	plotter.screenshot(filename=figname,
					   transparent_background=None,
					   return_img=True,
					   window_size=[fig_w,fig_h])
	plotter.close()






plotter.add_mesh(plt_pmjs, color=colors_list[i], point_size=6.0, render_points_as_spheres=True)


In [29]:
network_file=f'{mainpath}/1_seg/final_network_flip.vtk'
CS=pv.read(network_file)
print(CS)
print(CS.cell_data['elemTag'])
CS_upper = CS.threshold(value=(150, 157), scalars='elemTag')
CS_upper=CS_upper.extract_surface()
print(CS_upper.cell_data)

CS_upper.save(f'{mainpath}/1_seg/CS_upper.vtk')


PolyData (0x7a60fc7e1960)
  N Cells:    106196
  N Points:   80376
  N Strips:   0
  X Bounds:   -7.083e+01, 2.867e+01
  Y Bounds:   -1.500e+02, -5.102e+01
  Z Bounds:   3.270e+02, 4.363e+02
  N Arrays:   2
[150 150 150 ... 102 102 102]
pyvista DataSetAttributes
Association     : CELL
Active Scalars  : elemTag
Active Vectors  : fiber
Active Texture  : None
Active Normals  : None
Contains arrays :
    elemTag                 int32      (232,)               SCALARS
    fiber                   float32    (232, 3)             VECTORS
    vtkOriginalCellIds      int64      (232,)
